# Projeto Interconexão - Modelagem com Decision Tree Models

A documentação do projeto pode ser encontrada em [Modelo_Interconexão_BRAIN_Guilherme_Andrade](https://algarnet.sharepoint.com/sites/TriboAnalytics246/Documentos%20Compartilhados/Forms/AllItems.aspx?id=%2Fsites%2FTriboAnalytics246%2FDocumentos%20Compartilhados%2FGeneral%2FSquad%20Data%20Wizards%20%2D%20Projetos%2FModelo%5FInterconex%C3%A3o%5FBRAIN%5FGuilherme%5FAndrade&viewid=18eaba8b%2D6b2b%2D40e3%2D85d6%2De3c0e7598b0c&FolderCTID=0x012000E7FCBA090D95C348A4A8E9F05538F026)

Nesse *pipeline* serão realizadas as seguintes ações:

1. Amostragem dos dados
2. Extração das regras de interesse


In [ ]:
# === Bibliotecas necessárias
from snowflake.snowpark import Session
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import train_test_split
from sklearn import tree
from snowflake.snowpark.functions import col, lit, when
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier


# === Configurações
warnings.filterwarnings("ignore")
session = get_active_session()

In [ ]:
# === Função para a amostragem estratificada (snowpark)
def sample_by_class(df, class_value, fraction=0.10): #10% das amostras
    # Filtra pela classe e aplica amostragem
    return df.filter(col("SIPSTATUSCODE_135") == class_value).sample(fraction)

In [ ]:
# === Função para selecionar as colunas de interesse utilizando Snowpark
def altera_cols_snow(df):
    
    # Remove duplicatas
    df = df.drop_duplicates()

    # Cria coluna TARGET com expressão condicional
    df = df.with_column(
        "TARGET",
        when(col("ERRO_200_60MIN") > lit(0), lit(1)).otherwise(lit(0))
    )

    # Seleciona colunas desejadas
    df = df.select(
        "IDENTIFICATIONNUMBER_A_10",
        "IDENTIFICATIONNUMBER_B_11",
        "NUMBER_A_IP_10",
        "NUMBER_B_IP_11",
        "SIPSTATUSCODE_135",
        "ERRO_487_60MIN",
        "ERRO_200_60MIN",
        "ERRO_404_60MIN",
        "TARGET"
    )

    print("Colunas alteradas e retiradas!")

    # Substitui valores nulos por 0
    df = df.fillna(0)

    return df

# Processamento dos dados 

Esse processamento é necessário para rodas os modelos para extração de regras para os retornos SIPSTATUSCODE = 487 e 404

In [ ]:
# === Carregamento dos dados
df = session.table("DWDEV.DATASCIENCE.F_VOICETRAFFIC_SBC_ENGINEERED_1H")

# === Converter dados da coluna "SIPSTATUSCODE_135" (float) em int e depois string
df = df.withColumn("SIPSTATUSCODE_135", col("SIPSTATUSCODE_135").cast("int").cast("string"))
df

### Filtro para o cliente TELEMAP = "177.154.149.216"

In [ ]:
df_ip = df.filter(col("NUMBER_A_IP_10") == "177.154.149.216")
# === Contagem de ocorrências
contagem_filtro = df_ip.count()
print(f"Número de ocorrências com IP '177.154.149.216': {contagem_filtro}")

In [ ]:
from functools import reduce
# === Criação dos valores distintos presente na coluna "SIPSTATUSCODE_135" 
classes_list = [row["SIPSTATUSCODE_135"] for row in df_ip.select("SIPSTATUSCODE_135").distinct().collect()]

# === Usa a lista para gerar as amostras por valor
amostras = [sample_by_class(df_ip, c) for c in classes_list]

# === Une todas as amostras em um único DataFrame
df_amostrado = reduce(lambda df1, df2: df1.union(df2), amostras)

In [ ]:
#df_amostrado.save_as_table("PDF_SNOWPARK_1H", mode="overwrite")

In [ ]:
# === Carrega o dataframe novo
#df = session.table("DWDEV.DATASCIENCE.PDF_SNOWPARK_1H")
#df.show()

In [ ]:
df_snowpark = altera_cols_snow(df_amostrado)

In [ ]:
# === Verificar  quantidade de ocorrência dos códigos da coluna "SIPSTATUSCODE_135"
contagem = df_snowpark.groupBy("SIPSTATUSCODE_135").count()
filtrado_200 = contagem.filter(col("SIPSTATUSCODE_135") == '200')
filtrado_200.show()
filtrado_487 = contagem.filter(col("SIPSTATUSCODE_135") == '487')
filtrado_487.show()

In [ ]:
df_agregado = df_snowpark.select("ERRO_487_60MIN","ERRO_200_60MIN","TARGET")
df_agregado = df_agregado.filter(
    (col("ERRO_487_60MIN") != 0) |
    (col("ERRO_200_60MIN") != 0) |
    (col("TARGET") != 0)
)
df_agregado

In [ ]:
pdf_agregado = df_agregado.to_pandas()

In [ ]:
contagem_test = pdf_agregado.count()
print(f"Número de ocorrências com IP '177.154.149.216': {contagem_test}")

In [ ]:
X = pdf_agregado[["ERRO_487_60MIN"
                  #, "ERRO_200_60MIN"
                 ]]
y = pdf_agregado["TARGET"]

In [ ]:
print*

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size = 0.3,
                                                    random_state = 42,
                                                    stratify= y)

# Aplicar SMOTE apenas nos dados de treino
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)


tree_agregado = DecisionTreeClassifier(max_depth=2,
                                       class_weight="balanced",
                                       random_state=42)
tree_agregado.fit(X_train_res, y_train_res)

y_pred = tree_agregado.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

plt.figure(figsize =(8,5))
plot_tree(tree_agregado, feature_names = ["ERRO_487_60MIN","ERRO_200_60MIN"],
          class_names = ["Não atende", "Atende"], filled = True)
plt.show()